# Trace Analysis

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display  # noqa: A004
from trace_viewer import show_trace

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## Dashboard

In [ ]:
raw = pd.read_csv(TRACES_DIR / "results.csv")
raw["passed"] = raw["passed"].astype(bool)
for col in ["model_time_s", "tool_time_s"]:
    if col not in raw.columns:
        raw[col] = float("nan")
if "run_type" not in raw.columns:
    raw["run_type"] = ""

# Reruns (--filter) replace matching questions in the latest full run.
# Full runs are kept as-is for run-over-run history.
is_rerun = raw["run_type"] == "rerun"
full = raw[~is_rerun]
reruns = raw[is_rerun]

if not reruns.empty:
    drop_keys = set(zip(reruns["model"], reruns["trace_id"], strict=True))
    mask = full.apply(lambda r: (r["model"], r["trace_id"]) not in drop_keys, axis=1)
    full = full[mask]

results = pd.concat([full, reruns], ignore_index=True).sort_values("timestamp")
print(f"{len(results)} results across {results['model'].nunique()} models")

In [ ]:
# Run-over-run summary (last 10 model-runs, chronological)
runs = results.groupby(["timestamp", "model"]).agg(
    passed=("passed", "sum"),
    questions=("passed", "count"),
    pass_rate=("passed", "mean"),
    avg_tool_calls=("tool_calls", "mean"),
    avg_model_time=("model_time_s", "mean"),
    avg_tool_time=("tool_time_s", "mean"),
    avg_wall_time=("wall_time_s", "mean"),
    avg_tokens=("total_tokens", "mean"),
    avg_cost=("cost_usd", "mean"),
    total_cost=("cost_usd", "sum"),
).tail(10)
runs["pass_rate"] = runs["pass_rate"].map("{:.0%}".format)
runs["avg_tool_calls"] = runs["avg_tool_calls"].round(1)
runs["avg_model_time"] = runs["avg_model_time"].round(1)
runs["avg_tool_time"] = runs["avg_tool_time"].round(1)
runs["avg_wall_time"] = runs["avg_wall_time"].round(1)
runs["avg_tokens"] = runs["avg_tokens"].astype(int)
runs["avg_cost"] = runs["avg_cost"].map("${:.4f}".format)
runs["total_cost"] = runs["total_cost"].map("${:.4f}".format)
runs.index = runs.index.set_levels(
    runs.index.levels[1].str.replace("openrouter/", ""), level=1
)
runs

In [ ]:
# Select a run to inspect (default: latest)
RUN = results["timestamp"].max()

run = results[results["timestamp"] == RUN]
passed = run["passed"].sum()
total = len(run)
print(f"Selected: {RUN} | {passed}/{total} passed")

display_cols = ["workspace", "question", "passed", "tool_calls", "model_time_s", "tool_time_s", "wall_time_s", "failed_assertions"]
run[display_cols].sort_values("passed")

In [ ]:
# Failures for the selected run
failures = run[~run["passed"]]
if len(failures):
    print(f"{len(failures)} failures:")
    display(failures[["workspace", "question", "tool_calls", "failed_assertions", "error"]])
else:
    print("No failures!")

## Trace Viewer

In [ ]:
# Build trace index: model → [(label, path), ...]
trace_index: dict[str, list[tuple[str, Path]]] = {}
for model_dir in sorted(TRACES_DIR.iterdir()):
    if not model_dir.is_dir() or model_dir.name.endswith(".csv"):
        continue
    entries = []
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for f in sorted(ws_dir.glob("*.json")):
            data = json.loads(f.read_text())
            status = "PASS" if data["passed"] else "FAIL"
            entries.append((f"[{status}] {ws_dir.name} / {f.stem}", f))
    if entries:
        trace_index[model_dir.name] = entries

model_dropdown = widgets.Dropdown(
    options=list(trace_index.keys()),
    value=list(trace_index.keys())[-1],
    description="Model:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="60%"),
)
question_dropdown = widgets.Dropdown(
    description="Question:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="90%"),
)
output = widgets.Output()


def _update_questions(_change=None):
    entries = trace_index[model_dropdown.value]
    question_dropdown.options = [label for label, _ in entries]
    question_dropdown.value = question_dropdown.options[0]


def _show_trace(_change=None):
    entries = trace_index[model_dropdown.value]
    label_to_path = dict(entries)
    path = label_to_path.get(question_dropdown.value)
    if not path:
        return
    data = json.loads(path.read_text())
    with output:
        output.clear_output(wait=True)
        show_trace(data, max_chars=2000)


model_dropdown.observe(_update_questions, names="value")
question_dropdown.observe(_show_trace, names="value")

_update_questions()
display(model_dropdown, question_dropdown, output)
_show_trace()